# Sprint 04 – Modelagem Linear (MLAM)

**Integrantes:** (preencher nome completo e RM de cada integrante do grupo)

**Dataset:** Renewable Energy Production Dataset (2010–2020) — Kaggle

**Variável-alvo:** `Energy_Class` (Low / Medium / High)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score


## 1) Análise do dataset (3,0 pontos)

### a) Carregando o dataset

In [ ]:
# Carrega o dataset em um DataFrame
df = pd.read_csv('Renewable_Energy_Data.csv')

print(df.shape)
df.head()


### b) Encoding da variável-alvo (`Energy_Class`)

As classes de `Energy_Class` têm uma ordem natural: **Low < Medium < High**. Por isso, em vez de usar um `LabelEncoder` (que ordena por ordem alfabética e perderia esse sentido), o mapeamento foi feito manualmente, respeitando a ordem real da variável. Isso importa porque essa coluna numérica entra na matriz de correlação logo a seguir — uma codificação sem sentido de ordem geraria correlações sem significado.

In [ ]:
mapping = {'Low': 0, 'Medium': 1, 'High': 2}
df['Energy_Class_encoded'] = df['Energy_Class'].map(mapping)

df[['Energy_Class', 'Energy_Class_encoded']].head()


### c) Matriz de correlação

O dataset também tem três colunas categóricas nominais (`Region`, `Energy_Source`, `Season`). Correlação de Pearson só faz sentido para variáveis numéricas, então essas três colunas ficaram de fora da matriz — entram apenas as variáveis numéricas originais, junto com o alvo já codificado.

In [ ]:
numeric_features = [
    'Temperature_C', 'Wind_Speed_m_s', 'Solar_Radiation_kWh_m2',
    'Rainfall_mm', 'Efficiency_Ratio', 'Lagged_Production_MWh',
    'Combined_Weather_Index'
]

corr_matrix = df[numeric_features + ['Energy_Class_encoded']].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Matriz de correlação')
plt.show()


### d) Selecionar um subconjunto de features ou usar todas?

Olhando a linha/coluna `Energy_Class_encoded` da matriz:

- Nenhuma variável tem correlação forte com o alvo. A maior é `Efficiency_Ratio`, com cerca de **0.21** — uma correlação fraca.
- As demais variáveis (`Temperature_C`, `Wind_Speed_m_s`, `Solar_Radiation_kWh_m2`, `Rainfall_mm`, `Lagged_Production_MWh`, `Combined_Weather_Index`) têm correlação praticamente nula com o alvo (todas abaixo de 0.05 em módulo).
- Entre as próprias features, as correlações também são baixas (a maior, entre `Temperature_C` e `Efficiency_Ratio`, fica perto de -0.26) — ou seja, não há um par de variáveis redundantes (multicolinearidade) que justifique remover alguma delas.

**Decisão: manter todas as features.** Como nenhuma variável isolada se destaca como fortemente relevante, removê-la só pelo critério de correlação individual arriscaria descartar informação que só faz sentido em combinação com as outras — que é justamente o que um modelo consegue captar ao usar todas as entradas juntas. E como também não há colinearidade alta entre as features, não existe motivo técnico para eliminar nenhuma.

## 2) Definição e aplicação do modelo (4,0 pontos)

### a) e b) Modelo escolhido: Regressão Logística

A **Regressão Logística** (`LogisticRegression`) foi o modelo linear escolhido.

Ela é considerada **linear** porque sua fronteira de decisão (*decision boundary*) é definida por uma combinação linear das variáveis de entrada: para cada classe, o modelo calcula uma soma ponderada `w1*x1 + w2*x2 + ... + wn*xn + b` e depois transforma essa soma em probabilidade (via softmax, no caso de mais de 2 classes). Como essa soma é linear nos parâmetros `w`, as regiões que separam as classes no espaço das features são planos (hiperplanos) — não curvas — por isso o modelo é linear, mesmo classificando as 3 classes (`Low`, `Medium`, `High`).

In [ ]:
X = df[numeric_features]
y = df['Energy_Class_encoded']
class_labels = ['Low', 'Medium', 'High']


### c) e d) Treinamento nos dois cenários

Os dois cenários usam o **mesmo modelo, a mesma padronização (`StandardScaler`) e a mesma semente aleatória (`random_state=42`)** — a única diferença entre eles é a proporção treino/teste. Isso é proposital, para que a comparação dos resultados venha da diferença no tamanho dos conjuntos, e não de outra variação no experimento.

#### Cenário 1 — 60% treino / 40% teste

In [ ]:
X_train1, X_test1, y_train1, y_test1 = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

scaler1 = StandardScaler()
X_train1_scaled = scaler1.fit_transform(X_train1)
X_test1_scaled = scaler1.transform(X_test1)

model1 = LogisticRegression(max_iter=1000, random_state=42)
model1.fit(X_train1_scaled, y_train1)

y_pred1 = model1.predict(X_test1_scaled)

print('Treino:', X_train1.shape[0], 'amostras')
print('Teste:', X_test1.shape[0], 'amostras')


#### Cenário 2 — 85% treino / 15% teste

In [ ]:
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

scaler2 = StandardScaler()
X_train2_scaled = scaler2.fit_transform(X_train2)
X_test2_scaled = scaler2.transform(X_test2)

model2 = LogisticRegression(max_iter=1000, random_state=42)
model2.fit(X_train2_scaled, y_train2)

y_pred2 = model2.predict(X_test2_scaled)

print('Treino:', X_train2.shape[0], 'amostras')
print('Teste:', X_test2.shape[0], 'amostras')


## 3) Avaliação da performance (3,0 pontos)

### a) Matrizes de confusão

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm1 = confusion_matrix(y_test1, y_pred1)
ConfusionMatrixDisplay(cm1, display_labels=class_labels).plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Cenário 1 (60/40)')

cm2 = confusion_matrix(y_test2, y_pred2)
ConfusionMatrixDisplay(cm2, display_labels=class_labels).plot(ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title('Cenário 2 (85/15)')

plt.tight_layout()
plt.show()


### b) Acurácia, precisão e recall

Como são 3 classes com quantidades bem diferentes de exemplos (`High` tem bem mais amostras que `Low`), precisão e recall foram calculados com `average='weighted'` — isso pondera o resultado de cada classe pela quantidade de exemplos que ela tem, dando um resumo único e justo mesmo com classes desbalanceadas.

In [ ]:
resultados = pd.DataFrame({
    'Cenário': ['1 (60/40)', '2 (85/15)'],
    'Acurácia': [
        accuracy_score(y_test1, y_pred1),
        accuracy_score(y_test2, y_pred2)
    ],
    'Precisão': [
        precision_score(y_test1, y_pred1, average='weighted', zero_division=0),
        precision_score(y_test2, y_pred2, average='weighted', zero_division=0)
    ],
    'Recall': [
        recall_score(y_test1, y_pred1, average='weighted', zero_division=0),
        recall_score(y_test2, y_pred2, average='weighted', zero_division=0)
    ]
})

resultados


### c), d) e e) Discussão dos resultados

**c) O desempenho é adequado?**
Os dois cenários ficaram em torno de **79% de acurácia** (78–80%), com precisão e recall na mesma faixa. Para um problema de 3 classes (acerto ao acaso ≈ 33%), esse resultado é razoável, mas está longe de excelente. Nas matrizes de confusão, o modelo separa bem a classe `High` (a mais numerosa), mas confunde bastante `Low` com `Medium` — o que é esperado, já que nenhuma feature mostrou correlação forte com o alvo (a maior foi 0.21). Ou seja: o modelo aprendeu um padrão real, mas fraco.

**d) Comparando 40% x 15% de teste**
As métricas dos dois cenários ficaram muito parecidas (acurácia 0.79 vs 0.78, por exemplo). Isso indica que, para este dataset (1000 linhas), 600 amostras de treino já são suficientes para o modelo aprender praticamente o mesmo padrão que aprenderia com 850 — aumentar o treino não trouxe ganho perceptível de desempenho.

**e) Vantagens e limitações de um conjunto de teste maior ou menor**
- Um teste **maior** (40%, Cenário 1) dá uma avaliação mais confiável e estável do modelo, porque as métricas são calculadas sobre mais exemplos (400 aqui) — mas deixa menos dados disponíveis para o treino.
- Um teste **menor** (15%, Cenário 2) deixa mais dados para o treino (850 aqui), o que pode ajudar em datasets maiores ou mais complexos — mas a avaliação fica mais sujeita a variação aleatória, já que se baseia em poucos exemplos (só 150, divididos em 3 classes). Num dataset pequeno como este, um teste de 15% é arriscado: um único exemplo classificado errado já pesa bastante no resultado final.


## Conclusão

O modelo de Regressão Logística atingiu desempenho parecido (~78–79% de acurácia) nos dois cenários de treino/teste, mostrando que o tamanho do conjunto de treino, entre 60% e 85% dos dados, não foi o fator limitante — o teto de desempenho aqui vem da relação linear fraca entre as variáveis disponíveis e a classe de energia (`Energy_Class`).